In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import onnx
import onnxruntime as ort
from onnxsim import simplify
import numpy as np
from collections import Counter


# -----------------------------
# 1. Small CNN model for edge use
# -----------------------------
class SmallEdgeCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(),

            nn.Conv2d(8, 16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Linear(16, num_classes)

    def forward(self, x):
        x = self.features(x)

        zero = torch.tensor(0.0, device=x.device)
        one = torch.tensor(1.0, device=x.device)

        x = x + zero
        x = x * one

        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x   
    # def forward(self, x):
    #     x = self.features(x)
    #     x = torch.flatten(x, 1)
    #     x = self.classifier(x)
    #     return x


# -----------------------------
# 2. Create simple synthetic data
# -----------------------------
def make_synthetic_data(n_samples=1000):
    """
    Synthetic binary classification:
    class 0: bright square on left
    class 1: bright square on right
    Input shape: 1 x 16 x 16
    """
    X = torch.randn(n_samples, 1, 16, 16) * 0.1
    y = torch.randint(0, 2, (n_samples,))

    for i in range(n_samples):
        if y[i] == 0:
            X[i, :, 5:11, 2:7] += 1.0
        else:
            X[i, :, 5:11, 9:14] += 1.0

    return X, y


# -----------------------------
# 3. Train model briefly
# -----------------------------
def train_model():
    model = SmallEdgeCNN(num_classes=2)

    X_train, y_train = make_synthetic_data(1000)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    model.train()

    for epoch in range(10):
        optimizer.zero_grad()

        outputs = model(X_train)
        loss = criterion(outputs, y_train)

        loss.backward()
        optimizer.step()

        preds = outputs.argmax(dim=1)
        acc = (preds == y_train).float().mean().item()

        print(f"Epoch {epoch+1:02d} | Loss: {loss.item():.4f} | Acc: {acc:.4f}")

    return model


# -----------------------------
# 4. Export PyTorch model to ONNX
# -----------------------------
def export_to_onnx(model, onnx_path="edge_cnn.onnx"):
    model.eval()

    dummy_input = torch.randn(1, 1, 16, 16)

    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        export_params=True,
        opset_version=17,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["logits"],
        dynamic_axes={
            "input": {0: "batch_size"},
            "logits": {0: "batch_size"}
        }
    )

    print(f"\nExported ONNX model to: {onnx_path}")


# -----------------------------
# 5. Simplify ONNX model
# -----------------------------
def simplify_onnx_model(
    input_path="edge_cnn.onnx",
    output_path="edge_cnn_simplified.onnx"
):
    model = onnx.load(input_path)

    model_simplified, check = simplify(model)

    if not check:
        raise RuntimeError("Simplified ONNX model could not be validated.")

    onnx.save(model_simplified, output_path)

    print(f"Simplified ONNX model saved to: {output_path}")


# -----------------------------
# 6. Count ONNX node types
# -----------------------------
def inspect_onnx_nodes(path):
    model = onnx.load(path)
    node_types = [node.op_type for node in model.graph.node]
    counts = Counter(node_types)

    print(f"\nNode summary for {path}:")
    for op, count in counts.items():
        print(f"{op}: {count}")

    print(f"Total nodes: {len(node_types)}")


# -----------------------------
# 7. Compare PyTorch and ONNX outputs
# -----------------------------
def verify_outputs(model, onnx_path="edge_cnn_simplified.onnx"):
    model.eval()

    test_input = torch.randn(4, 1, 16, 16)

    with torch.no_grad():
        torch_output = model(test_input).numpy()

    session = ort.InferenceSession(
        onnx_path,
        providers=["CPUExecutionProvider"]
    )

    onnx_output = session.run(
        ["logits"],
        {"input": test_input.numpy()}
    )[0]

    max_abs_diff = np.max(np.abs(torch_output - onnx_output))

    print("\nVerification:")
    print("PyTorch output shape:", torch_output.shape)
    print("ONNX output shape:", onnx_output.shape)
    print("Max absolute difference:", max_abs_diff)


# -----------------------------
# Main
# -----------------------------
if __name__ == "__main__":
    model = train_model()

    export_to_onnx(model, "edge_cnn.onnx")

    inspect_onnx_nodes("edge_cnn.onnx")

    simplify_onnx_model(
        input_path="edge_cnn.onnx",
        output_path="edge_cnn_simplified.onnx"
    )

    inspect_onnx_nodes("edge_cnn_simplified.onnx")

    verify_outputs(model, "edge_cnn_simplified.onnx")

Epoch 01 | Loss: 0.6997 | Acc: 0.5050
Epoch 02 | Loss: 0.6981 | Acc: 0.5050
Epoch 03 | Loss: 0.6968 | Acc: 0.5050
Epoch 04 | Loss: 0.6955 | Acc: 0.5050
Epoch 05 | Loss: 0.6944 | Acc: 0.5050
Epoch 06 | Loss: 0.6934 | Acc: 0.5050
Epoch 07 | Loss: 0.6926 | Acc: 0.5050
Epoch 08 | Loss: 0.6918 | Acc: 0.5050
Epoch 09 | Loss: 0.6912 | Acc: 0.5050
Epoch 10 | Loss: 0.6907 | Acc: 0.5050

Exported ONNX model to: edge_cnn.onnx

Node summary for edge_cnn.onnx:
Conv: 2
Relu: 2
GlobalAveragePool: 1
Constant: 2
Add: 1
Mul: 1
Flatten: 1
Gemm: 1
Total nodes: 11
Simplified ONNX model saved to: edge_cnn_simplified.onnx

Node summary for edge_cnn_simplified.onnx:
Conv: 2
Relu: 2
GlobalAveragePool: 1
Flatten: 1
Gemm: 1
Total nodes: 7

Verification:
PyTorch output shape: (4, 2)
ONNX output shape: (4, 2)
Max absolute difference: 8.940697e-08


/var/folders/ql/yg6ll6yn02vcx3270zpjjz6h0000gn/T/ipykernel_25051/3410503506.py:35: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  zero = torch.tensor(0.0, device=x.device)
/var/folders/ql/yg6ll6yn02vcx3270zpjjz6h0000gn/T/ipykernel_25051/3410503506.py:36: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  one = torch.tensor(1.0, device=x.device)
